# Clonar TU voz a Piper — 2 celdas
Guion = `corpus/corpus_es_1300.txt`. Grabá un clip por frase: `f00001.wav` = frase 1, etc.
Comprimí la carpeta `wavs/` en **dataset.zip**. Entrenamiento con **piper1-gpl** (receta probada).

**Antes:** Entorno de ejecución → Cambiar tipo → **T4 GPU**.
**Celda 1**: subís el zip. **Celda 2**: entrena.

In [ ]:
#@title 1. SUBIR TUS AUDIOS Y PREPARAR — subís dataset.zip — ~15-20 min
import torch, os, re, wave, glob, zipfile
import numpy as np
assert torch.cuda.is_available(), "Activá T4: Entorno de ejecución -> Cambiar tipo de entorno -> GPU"
print("GPU:", torch.cuda.get_device_name(0))
!wget -q -O /content/corpus.txt "https://raw.githubusercontent.com/Lazy-Money/Loud-Web/claude/readvox-research-slumjx/colab/corpus/corpus_es_1300.txt"
CORPUS = [l.strip() for l in open("/content/corpus.txt", encoding="utf-8") if l.strip()]
from google.colab import files
print("Subí tu dataset.zip (carpeta wavs/ con f00001.wav = frase 1, etc.)")
up = files.upload()
with zipfile.ZipFile(list(up.keys())[0]) as z: z.extractall("/content/unzip")
wavs = glob.glob("/content/unzip/**/*.wav", recursive=True)
print(f"{len(wavs)} audios. Emparejando y normalizando...")
def normw(src, dst):
    with wave.open(src,"rb") as w:
        n,r,ch,sw = w.getnframes(),w.getframerate(),w.getnchannels(),w.getsampwidth()
        raw = w.readframes(n)
    d = np.frombuffer(raw, dtype={1:np.int8,2:np.int16,4:np.int32}[sw]).astype(np.float32)
    if sw!=2: d = d/(2**(8*sw-1))*32767
    if ch>1: d = d.reshape(-1,ch).mean(axis=1)
    if r!=22050:
        idx = np.linspace(0,len(d)-1,int(len(d)*22050/r)); d = np.interp(idx,np.arange(len(d)),d)
    d = np.clip(d,-32768,32767).astype(np.int16)
    with wave.open(dst,"wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050); w.writeframes(d.tobytes())
    return len(d)/22050
os.makedirs("/content/dataset/wavs", exist_ok=True)
rows, total, skip = [], 0.0, 0
for src in sorted(wavs):
    m = re.search(r"f?(\d{3,6})", os.path.basename(src))
    if not m or not (1<=int(m.group(1))<=len(CORPUS)): skip+=1; continue
    idx = int(m.group(1)); n = f"f{idx:05d}.wav"
    seg = normw(src, f"/content/dataset/wavs/{n}")
    if not 1.0<=seg<=20.0: skip+=1; continue
    rows.append(f"{n}|{CORPUS[idx-1]}"); total+=seg
open("/content/dataset/metadata.csv","w",encoding="utf-8").write("\n".join(rows)+"\n")
assert rows, "Ningún audio se pudo emparejar. Deben llamarse f00001.wav, f00002.wav..."
print(f"Dataset: {len(rows)} clips, {total/60:.1f} min de tu voz.")

print("Instalando Piper (piper1-gpl, receta probada)...")
!apt-get -q update -y > /dev/null 2>&1
!apt-get -q install -y build-essential cmake ninja-build espeak-ng > /dev/null 2>&1
%cd /content
!git clone -q https://github.com/OHF-voice/piper1-gpl.git
%cd /content/piper1-gpl
!pip install -q -e '.[train]'
!pip install -q --upgrade scikit-build "protobuf==3.20.3"
!bash build_monotonic_align.sh > /dev/null 2>&1
!python setup.py build_ext --inplace > /dev/null 2>&1
!wget -q -nc -O /content/base.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_ES/davefx/medium/epoch%3D2218-step%3D562840.ckpt"
print("\n=== LISTO. Ahora corré la celda 2 para entrenar. ===")

In [ ]:
#@title 2. ENTRENAR Y DESCARGAR — 2-4 h (podés cortarla y reejecutarla para exportar lo entrenado)
import glob, os
def last_ckpt():
    c = sorted(glob.glob("/content/train_out/**/*.ckpt", recursive=True), key=os.path.getmtime)
    return c[-1] if c else None
if not last_ckpt():
    !cd /content/piper1-gpl && python -m piper.train fit \
      --data.voice_name "mi_voz" \
      --data.csv_path /content/dataset/metadata.csv \
      --data.audio_dir /content/dataset/wavs \
      --data.espeak_voice es \
      --data.cache_dir /content/cache \
      --data.config_path /content/mi_voz.onnx.json \
      --data.batch_size 16 \
      --model.sample_rate 22050 \
      --data.validation_split 0 --data.num_test_examples 0 \
      --trainer.default_root_dir /content/train_out \
      --trainer.accelerator gpu --trainer.devices 1 \
      --trainer.max_epochs 1000 \
      --trainer.precision 16-mixed \
      --checkpoint.every_n_epochs 25 --checkpoint.save_top_k -1 --checkpoint.monitor null \
      --ckpt_path /content/base.ckpt
ck = last_ckpt()
assert ck, "No hay checkpoint todavía: dejá entrenar unos minutos (se guarda cada 25 epochs) y reejecutá esta celda."
!cd /content/piper1-gpl && python -m piper.train.export_onnx --checkpoint "{ck}" --output-file /content/mi_voz.onnx
from google.colab import files
files.download("/content/mi_voz.onnx")
files.download("/content/mi_voz.onnx.json")
print("Copiá ambos a tu carpeta de voces de LoudVox y elegí 'mi_voz' en Configuración")